# Product Retention & Cohort Analysis

**Author:** Product Analyst Portfolio Project
**Data:** 3,000 users, 65,000+ events, Jan 2023 - Sep 2024

This notebook demonstrates end-to-end cohort retention analysis for a SaaS product.

In [ ]:
# CELL 1 — Imports & Config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Define paths
DATA_PATH = '../data/user_events.csv'
CHARTS_PATH = '../charts/'

# Color palette for cohort heatmap (blue = good retention, red = poor retention)
COHORT_CMAP = sns.diverging_palette(250, 10, s=85, l=55, as_cmap=True)

# Ensure charts directory exists
os.makedirs(CHARTS_PATH, exist_ok=True)

print("=" * 60)
print("PRODUCT RETENTION & COHORT ANALYSIS")
print("=" * 60)
print(f"Data path: {DATA_PATH}")
print(f"Charts path: {CHARTS_PATH}")
print("\nConfiguration loaded successfully ✓")

## CELL 2 — Load & Validate Data

In [ ]:
# CELL 2 — Load & Validate
print("\n" + "=" * 60)
print("CELL 2: LOAD & VALIDATE DATA")
print("=" * 60)

# Load the dataset
df = pd.read_csv(DATA_PATH)

# Parse dates
df['registration_date'] = pd.to_datetime(df['registration_date'])
df['event_date'] = pd.to_datetime(df['event_date'])

# Validate columns
expected_columns = ['user_id', 'registration_date', 'event_date', 'event_type', 
                    'revenue', 'plan_type', 'acquisition_channel']
missing_cols = set(expected_columns) - set(df.columns)
if missing_cols:
    print(f"❌ Missing columns: {missing_cols}")
else:
    print("✅ All expected columns present")

# Print summary statistics
print("\n📊 DATASET OVERVIEW:")
print(f"   Total events: {len(df):,}")
print(f"   Unique users: {df['user_id'].nunique():,}")
print(f"   Date range: {df['event_date'].min().strftime('%Y-%m-%d')} to {df['event_date'].max().strftime('%Y-%m-%d')}")
print(f"   Registration range: {df['registration_date'].min().strftime('%Y-%m-%d')} to {df['registration_date'].max().strftime('%Y-%m-%d')}")

print("\n📋 EVENT TYPE BREAKDOWN:")
event_counts = df['event_type'].value_counts()
for event, count in event_counts.items():
    pct = count / len(df) * 100
    print(f"   {event:20s}: {count:5,} ({pct:5.1f}%)")

print("\n💰 REVENUE SUMMARY:")
print(f"   Total revenue: ${df['revenue'].sum():,.2f}")
print(f"   Revenue events: {len(df[df['revenue'] > 0]):,}")
print(f"   Avg revenue per event: ${df[df['revenue'] > 0]['revenue'].mean():.2f}")

# Get unique users dataframe for cohort analysis
users_df = df.drop_duplicates('user_id').copy()
print(f"\n👥 USER ATTRIBUTES:")
print(f"   Unique users: {len(users_df):,}")
print(f"   Channels: {users_df['acquisition_channel'].value_counts().to_dict()}")
print(f"   Plans: {users_df['plan_type'].value_counts().to_dict()}")

## CELL 3 — Cohort Construction

In [ ]:
# CELL 3 — Cohort Construction
print("\n" + "=" * 60)
print("CELL 3: COHORT CONSTRUCTION")
print("=" * 60)

# Assign each user to a cohort month based on registration date
users_df['cohort_month'] = users_df['registration_date'].dt.to_period('M')

# Compute cohort sizes
cohort_sizes = users_df.groupby('cohort_month').size().reset_index(name='cohort_size')
print("\n📅 COHORT SIZES BY MONTH:")
for _, row in cohort_sizes.iterrows():
    print(f"   {row['cohort_month']}: {row['cohort_size']:,} users")

# Calculate periods since registration for each event
df['periods'] = (df['event_date'] - df['registration_date']).dt.days // 30
df['cohort_month'] = df['registration_date'].dt.to_period('M')

# Get unique active periods per user (for retention calculation)
user_activity = df[df['event_type'].isin(['return_visit', 'first_purchase', 'upgrade'])].copy()
user_activity = user_activity.groupby(['user_id', 'cohort_month', 'periods']).size().reset_index(name='activity_count')

print(f"\n✅ Cohorts assigned to {len(users_df):,} users")
print(f"✅ Activity tracked across {user_activity['periods'].max() + 1} periods (0-{user_activity['periods'].max()} months)")

## CELL 4 — Build Retention Table

In [ ]:
# CELL 4 — Retention Table
print("\n" + "=" * 60)
print("CELL 4: BUILD RETENTION TABLE")
print("=" * 60)

# Create the cohort retention matrix
def get_retention_matrix():
    # Get all active user-period combinations
    active_users = user_activity.groupby(['cohort_month', 'periods'])['user_id'].nunique().reset_index()
    active_users = active_users.rename(columns={'user_id': 'active_users'})
    
    # Merge with cohort sizes
    cohort_data = active_users.merge(cohort_sizes, on='cohort_month')
    
    # Calculate retention percentage
    cohort_data['retention_pct'] = (cohort_data['active_users'] / cohort_data['cohort_size'] * 100).round(1)
    
    # Create pivot table
    retention_matrix = cohort_data.pivot_table(
        index='cohort_month',
        columns='periods',
        values='retention_pct',
        fill_value=0
    )
    
    return retention_matrix, cohort_data

retention_matrix, cohort_data = get_retention_matrix()

print("\n📊 COHORT RETENTION MATRIX (%):")
print("-" * 80)

# Format and display the retention matrix
formatted_matrix = retention_matrix.copy()
formatted_matrix.index = formatted_matrix.index.astype(str)
formatted_matrix.columns = [f"M{int(c)}" for c in formatted_matrix.columns]

# Print the matrix
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(formatted_matrix.to_string())

# Print cohort size row
print("-" * 80)
cohort_size_dict = dict(zip(cohort_sizes['cohort_month'].astype(str), cohort_sizes['cohort_size']))
print("Cohort Size: ", end="")
for cohort in formatted_matrix.index:
    print(f"{cohort_size_dict.get(cohort, 0):>6} ", end="")
print()

print("\n✅ Retention matrix built successfully")
print(f"   {len(retention_matrix)} cohorts x {len(retention_matrix.columns)} periods")

## CELL 5 — Retention Heatmap

In [ ]:
# CELL 5 — Retention Heatmap
print("\n" + "=" * 60)
print("CELL 5: RETENTION HEATMAP")
print("=" * 60)

# Create the heatmap
fig, ax = plt.subplots(figsize=(14, 10))

# Prepare data for heatmap
heatmap_data = retention_matrix.copy()
heatmap_data.index = heatmap_data.index.astype(str)

# Create heatmap with annotations
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1f',
    cmap=COHORT_CMAP,
    center=25,
    vmin=0,
    vmax=100,
    cbar_kws={'label': 'Retention %'},
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 9}
)

# Customize labels
ax.set_xlabel('Months Since Registration', fontsize=12, fontweight='bold')
ax.set_ylabel('Cohort Month', fontsize=12, fontweight='bold')
ax.set_title('Monthly Cohort Retention Heatmap\n(% of Original Cohort Still Active)', 
             fontsize=14, fontweight='bold', pad=20)

# Rotate x-axis labels
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.tight_layout()

# Save the chart
chart_path = CHARTS_PATH + 'cohort_heatmap.png'
plt.savefig(chart_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ Heatmap saved to: {chart_path}")

plt.show()

print("\n💡 INSIGHT: Reading down a column shows if retention is improving over time.")
print("   Later cohorts (bottom rows) generally show better retention, especially at M1-M3.")

## CELL 6 — Retention Curves

In [ ]:
# CELL 6 — Retention Curves
print("\n" + "=" * 60)
print("CELL 6: RETENTION CURVES")
print("=" * 60)

# Plot retention curves for the first 6 cohorts
fig, ax = plt.subplots(figsize=(14, 8))

# Select first 6 cohorts with enough data
cohorts_to_plot = retention_matrix.index[:6]
colors = sns.color_palette("husl", len(cohorts_to_plot))

best_cohort = None
best_retention_m3 = 0
worst_cohort = None
worst_retention_m3 = 100

for i, cohort in enumerate(cohorts_to_plot):
    cohort_data = retention_matrix.loc[cohort].dropna()
    months = cohort_data.index.astype(int)
    retention_values = cohort_data.values
    
    # Track best and worst at month 3
    if 3 in months:
        m3_retention = cohort_data[3]
        if m3_retention > best_retention_m3:
            best_retention_m3 = m3_retention
            best_cohort = cohort
        if m3_retention < worst_retention_m3:
            worst_retention_m3 = m3_retention
            worst_cohort = cohort
    
    ax.plot(months, retention_values, marker='o', linewidth=2.5, 
            label=str(cohort), color=colors[i], markersize=6)

# Add vertical lines for Day 7 and Day 30 (convert to months approximation)
ax.axvline(x=0.23, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Day 7')
ax.axvline(x=1.0, color='gray', linestyle=':', linewidth=1.5, alpha=0.7, label='Day 30')

# Annotate best and worst cohorts
if best_cohort:
    ax.annotate(f'Best: {best_cohort}\n({best_retention_m3:.1f}% at M3)',
                xy=(3, best_retention_m3), xytext=(4.5, best_retention_m3 + 10),
                fontsize=10, fontweight='bold', color='green',
                arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

if worst_cohort:
    ax.annotate(f'Worst: {worst_cohort}\n({worst_retention_m3:.1f}% at M3)',
                xy=(3, worst_retention_m3), xytext=(4.5, worst_retention_m3 - 15),
                fontsize=10, fontweight='bold', color='red',
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_xlabel('Months Since Registration', fontsize=12, fontweight='bold')
ax.set_ylabel('Retention %', fontsize=12, fontweight='bold')
ax.set_title('Retention Curves by Cohort\n(First 6 Cohorts)', fontsize=14, fontweight='bold', pad=20)
ax.legend(title='Cohort', loc='upper right', bbox_to_anchor=(1.15, 1))
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save the chart
chart_path = CHARTS_PATH + 'retention_curve.png'
plt.savefig(chart_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ Retention curves saved to: {chart_path}")

plt.show()

print(f"\n📈 RETENTION CURVE INSIGHTS:")
print(f"   Best cohort at Month 3: {best_cohort} ({best_retention_m3:.1f}%)")
print(f"   Worst cohort at Month 3: {worst_cohort} ({worst_retention_m3:.1f}%)")
print(f"   Difference: {best_retention_m3 - worst_retention_m3:.1f} percentage points")

## CELL 7 — LTV per Cohort

In [ ]:
# CELL 7 — LTV per Cohort
print("\n" + "=" * 60)
print("CELL 7: LTV PER COHORT")
print("=" * 60)

# Calculate cumulative revenue per cohort
revenue_by_cohort = df.groupby('cohort_month')['revenue'].sum().reset_index()
revenue_by_cohort = revenue_by_cohort.merge(cohort_sizes, on='cohort_month')
revenue_by_cohort['ltv'] = revenue_by_cohort['revenue'] / revenue_by_cohort['cohort_size']

# Sort by cohort month
revenue_by_cohort = revenue_by_cohort.sort_values('cohort_month')

print("\n💰 LTV BY COHORT:")
print("-" * 50)
print(f"{'Cohort':<12} {'Users':<8} {'Revenue':<12} {'LTV':<10}")
print("-" * 50)

for _, row in revenue_by_cohort.iterrows():
    print(f"{str(row['cohort_month']):<12} {row['cohort_size']:<8} ${row['revenue']:<11,.0f} ${row['ltv']:<10.2f}")

# Get top 3 cohorts by LTV
top_3_ltv = revenue_by_cohort.nlargest(3, 'ltv')
print("\n🏆 TOP 3 COHORTS BY LTV:")
for i, (_, row) in enumerate(top_3_ltv.iterrows(), 1):
    print(f"   {i}. {row['cohort_month']}: ${row['ltv']:.2f} per user ({row['cohort_size']} users)")

# Calculate why top cohorts are better
print("\n🔍 WHY TOP COHORTS PERFORM BETTER:")
avg_ltv = revenue_by_cohort['ltv'].mean()
print(f"   Average LTV across all cohorts: ${avg_ltv:.2f}")
print(f"   Top cohort LTV premium: ${top_3_ltv.iloc[0]['ltv'] - avg_ltv:.2f} ({(top_3_ltv.iloc[0]['ltv'] / avg_ltv - 1) * 100:.1f}% higher)")

# Create bar chart
fig, ax = plt.subplots(figsize=(14, 7))

cohorts = revenue_by_cohort['cohort_month'].astype(str)
ltv_values = revenue_by_cohort['ltv']

# Color bars by LTV relative to average
colors = ['#2ecc71' if v > avg_ltv else '#e74c3c' for v in ltv_values]

bars = ax.bar(cohorts, ltv_values, color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, value in zip(bars, ltv_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'${value:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Add average line
ax.axhline(y=avg_ltv, color='blue', linestyle='--', linewidth=2, label=f'Average LTV (${avg_ltv:.0f})')

ax.set_xlabel('Cohort Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Lifetime Value ($)', fontsize=12, fontweight='bold')
ax.set_title('LTV by Cohort\n(Green = Above Average, Red = Below Average)', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend()
ax.set_ylim(0, max(ltv_values) * 1.15)

plt.xticks(rotation=45)
plt.tight_layout()

# Save the chart
chart_path = CHARTS_PATH + 'ltv_by_cohort.png'
plt.savefig(chart_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✅ LTV chart saved to: {chart_path}")

plt.show()

## CELL 8 — Channel Analysis

In [ ]:
# CELL 8 — Channel Analysis
print("\n" + "=" * 60)
print("CELL 8: ACQUISITION CHANNEL ANALYSIS")
print("=" * 60)

def calculate_retention_by_channel():
    results = []
    
    for channel in users_df['acquisition_channel'].unique():
        channel_users = users_df[users_df['acquisition_channel'] == channel]['user_id'].tolist()
        
        # Get activity for these users
        channel_activity = df[df['user_id'].isin(channel_users)].copy()
        channel_activity['days_since_reg'] = (channel_activity['event_date'] - 
                                               channel_activity['registration_date']).dt.days
        
        total_users = len(channel_users)
        
        # Day 7 retention
        day7_active = channel_activity[channel_activity['days_since_reg'] <= 7]['user_id'].nunique()
        day7_retention = day7_active / total_users * 100
        
        # Day 30 retention
        day30_active = channel_activity[channel_activity['days_since_reg'] <= 30]['user_id'].nunique()
        day30_retention = day30_active / total_users * 100
        
        # LTV for this channel
        channel_ltv = df[df['user_id'].isin(channel_users)]['revenue'].sum() / total_users
        
        results.append({
            'channel': channel,
            'users': total_users,
            'day_7_retention': day7_retention,
            'day_30_retention': day30_retention,
            'ltv': channel_ltv
        })
    
    return pd.DataFrame(results)

channel_analysis = calculate_retention_by_channel()
channel_analysis = channel_analysis.sort_values('day_30_retention', ascending=False)

print("\n📊 RETENTION BY ACQUISITION CHANNEL:")
print("-" * 80)
print(f"{'Channel':<15} {'Users':<8} {'Day 7 Ret':<12} {'Day 30 Ret':<12} {'LTV':<10}")
print("-" * 80)

for _, row in channel_analysis.iterrows():
    print(f"{row['channel']:<15} {row['users']:<8} {row['day_7_retention']:<11.1f}% "
          f"{row['day_30_retention']:<11.1f}% ${row['ltv']:<10.2f}")

best_channel = channel_analysis.iloc[0]
worst_channel = channel_analysis.iloc[-1]

print("\n📈 CHANNEL PERFORMANCE COMPARISON:")
print(f"   Best channel (30-day retention): {best_channel['channel']} ({best_channel['day_30_retention']:.1f}%)")
print(f"   Worst channel (30-day retention): {worst_channel['channel']} ({worst_channel['day_30_retention']:.1f}%)")
print(f"   Difference: {best_channel['day_30_retention'] - worst_channel['day_30_retention']:.1f} percentage points "
      f"({(best_channel['day_30_retention'] / worst_channel['day_30_retention'] - 1) * 100:.1f}% relative)")

## CELL 9 — Onboarding Impact

In [ ]:
# CELL 9 — Onboarding Impact
print("\n" + "=" * 60)
print("CELL 9: ONBOARDING IMPACT ANALYSIS")
print("=" * 60)

# Identify users who completed onboarding
onboarding_completers = set(df[df['event_type'] == 'onboarding_complete']['user_id'].unique())

onboarding_users = users_df[users_df['user_id'].isin(onboarding_completers)].copy()
non_onboarding_users = users_df[~users_df['user_id'].isin(onboarding_completers)].copy()

print(f"\n👥 ONBOARDING COMPLETION BREAKDOWN:")
print(f"   Completed onboarding: {len(onboarding_users):,} users ({len(onboarding_users)/len(users_df)*100:.1f}%)")
print(f"   Did not complete: {len(non_onboarding_users):,} users ({len(non_onboarding_users)/len(users_df)*100:.1f}%)")

def calculate_retention_for_group(user_ids):
    if len(user_ids) == 0:
        return {'day_7': 0, 'day_30': 0}
    
    group_activity = df[df['user_id'].isin(user_ids)].copy()
    group_activity['days_since_reg'] = (group_activity['event_date'] - 
                                         group_activity['registration_date']).dt.days
    
    total = len(user_ids)
    day7_active = group_activity[group_activity['days_since_reg'] <= 7]['user_id'].nunique()
    day30_active = group_activity[group_activity['days_since_reg'] <= 30]['user_id'].nunique()
    
    return {
        'day_7': day7_active / total * 100,
        'day_30': day30_active / total * 100
    }

onboarding_retention = calculate_retention_for_group(onboarding_users['user_id'].tolist())
non_onboarding_retention = calculate_retention_for_group(non_onboarding_users['user_id'].tolist())

print(f"\n📊 RETENTION COMPARISON:")
print("-" * 60)
print(f"{'Group':<25} {'Day 7':<15} {'Day 30':<15}")
print("-" * 60)
print(f"{'Onboarding Completed':<25} {onboarding_retention['day_7']:<14.1f}% {onboarding_retention['day_30']:<14.1f}%")
print(f"{'Onboarding Not Completed':<25} {non_onboarding_retention['day_7']:<14.1f}% {non_onboarding_retention['day_30']:<14.1f}%")

day7_lift = ((onboarding_retention['day_7'] / non_onboarding_retention['day_7']) - 1) * 100
day30_lift = ((onboarding_retention['day_30'] / non_onboarding_retention['day_30']) - 1) * 100
abs_day30_lift = onboarding_retention['day_30'] - non_onboarding_retention['day_30']

print(f"\n🚀 RETENTION LIFT FROM ONBOARDING:")
print(f"   Day 30: +{abs_day30_lift:.1f} percentage points ({day30_lift:+.0f}% relative)")

current_completion_rate = len(onboarding_users) / len(users_df)
print(f"\n💡 BUSINESS IMPACT ESTIMATE:")
print(f"   Current onboarding completion: {current_completion_rate*100:.1f}%")
print(f"   If improved by 25% → +{abs_day30_lift * 0.25:.1f}pp to 30-day retention")

## CELL 10 — Key Insights & PM Summary

In [ ]:
# CELL 10 — Key Insights & PM Summary
print("\n" + "=" * 70)
print("CELL 10: KEY INSIGHTS & PM SUMMARY")
print("=" * 70)

print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    KEY INSIGHTS (WITH NUMBERS)                       ║
╚══════════════════════════════════════════════════════════════════════╝
""")

print(f"""
1. ONBOARDING COMPLETION IS THE STRONGEST RETENTION PREDICTOR
   → Users who complete onboarding: {onboarding_retention['day_30']:.1f}% 30-day retention
   → Users who skip onboarding: {non_onboarding_retention['day_30']:.1f}% 30-day retention
   → Retention lift: +{day30_lift:.0f}% relative improvement
   → Current completion rate: {current_completion_rate*100:.1f}%

2. ACQUISITION CHANNEL QUALITY VARIES SIGNIFICANTLY
   → Best channel: {best_channel['channel']} ({best_channel['day_30_retention']:.1f}% retention, ${best_channel['ltv']:.0f} LTV)
   → Worst channel: {worst_channel['channel']} ({worst_channel['day_30_retention']:.1f}% retention, ${worst_channel['ltv']:.0f} LTV)
   → Gap: {best_channel['day_30_retention'] - worst_channel['day_30_retention']:.1f}pp difference in retention

3. RECOMMENDED EXPERIMENTS (A/B TESTS)
   → Onboarding Progress Bar: Expected +{abs_day30_lift * 0.20:.1f}pp to 30-day retention
   → Referral Incentive Program: Shift acquisition mix to higher-LTV users
   → Day-7 Re-engagement Email: Prevent early churn with targeted outreach
""")

print("""
╔══════════════════════════════════════════════════════════════════════╗
║          HOW TO PRESENT THIS TO A PM                                 ║
╚══════════════════════════════════════════════════════════════════════╝

📌 THE NARRATIVE ARC:

1. "START WITH THE BOTTOM LINE"
   "30-day retention is improving, but onboarding completion remains "
   "our biggest untapped lever."

2. "SHOW THE HEATMAP"
   "This chart tells the story. Later cohorts retain better than "
   "early ones, especially at Month 1 and Month 2."

3. "INTRODUCE THE KEY EXPERIMENT"
   "Users who complete onboarding have {day30_lift:.0f}% better 30-day retention."
   "If we improve completion by 25%, we expect a {abs_day30_lift * 0.25:.1f}pp jump."

4. "ADDRESS ACQUISITION"
   "{best_channel['channel']} users are worth {(best_channel['ltv'] / worst_channel['ltv']):.1f}x "
   "{worst_channel['channel']} users. We should shift our mix."

5. "CLOSE WITH CLEAR NEXT STEPS"
   "Prioritize the onboarding progress bar test this sprint — "
   "highest confidence, highest impact, easiest to implement."
""")

print("\n✅ ANALYSIS COMPLETE")
print(f"📁 Run the cells above to generate charts in: {CHARTS_PATH}")
print("\n" + "=" * 70)